In [ ]:
# Cell 13: Inference dengan TTA (Test Time Augmentation)
import pandas as pd  # FIXED: Import pandas
from PIL import Image as PILImage

@torch.no_grad()
def predict_with_tta(model, image, tta_transform, device, use_flip=True):
    """
    Predict with Test Time Augmentation
    Args:
        image: numpy array (H, W, C) in RGB, values 0-255
    """
    model.eval()
    
    # Original - langsung transform tanpa denormalize ulang
    orig = tta_transform(image=image)['image'].unsqueeze(0).to(device)
    logits = model(orig)
    probs = F.softmax(logits, dim=1)
    
    if use_flip:
        # Flipped version
        image_flip = np.fliplr(image).copy()
        flipped = tta_transform(image=image_flip)['image'].unsqueeze(0).to(device)
        logits_flip = model(flipped)
        probs_flip = F.softmax(logits_flip, dim=1)
        # Flip back probabilities
        probs_flip = torch.flip(probs_flip, dims=[-1])
        # Ensemble average
        probs = (probs + probs_flip) / 2
    
    return torch.argmax(probs, dim=1).squeeze(0).cpu().numpy()

def generate_submission(model, test_img_dir, test_ids, device, use_tta=True, 
                        img_size=(640, 480)):
    """
    Generate submission.csv file
    Args:
        test_img_dir: path to test images directory
        test_ids: list of test image IDs (without .jpg)
        device: torch device
        use_tta: whether to use test time augmentation
        img_size: original image size (width, height)
    """
    model.eval()
    results = []
    
    # Create transform for inference (resize + normalize)
    infer_transform = A.Compose([
        A.Resize(512, 512),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])
    
    print(f"Generating predictions for {len(test_ids)} images...")
    
    for img_id in tqdm(test_ids, desc="Inference"):
        # FIXED: Load raw image directly from disk
        img_path = os.path.join(test_img_dir, f"{img_id}.jpg")
        img_np = np.array(PILImage.open(img_path).convert('RGB'))
        
        # Predict
        if use_tta:
            mask = predict_with_tta(
                model, img_np, infer_transform, device, use_flip=True
            )
        else:
            transformed = infer_transform(image=img_np)['image'].unsqueeze(0).to(device)
            logits = model(transformed)
            mask = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()
        
        # FIXED: Use Image.Resampling.NEAREST (Pillow >= 10.0)
        mask_img = PILImage.fromarray(mask.astype(np.uint8))
        # Check Pillow version for compatibility
        try:
            # Pillow >= 10.0
            mask_img = mask_img.resize(img_size, PILImage.Resampling.NEAREST)
        except AttributeError:
            # Pillow < 10.0 (fallback)
            mask_img = mask_img.resize(img_size, Image.NEAREST)
        mask = np.array(mask_img)
        
        # Convert to RLE per class
        rles = mask_to_rle(mask, empty_classes=EMPTY_CLASSES)
        
        for class_id in range(NUM_CLASSES):
            results.append({
                'id': f"{img_id}_{class_id}",
                'encoded_pixels': rles[class_id]
            })
    
    # Create DataFrame and save
    df = pd.DataFrame(results)
    df.to_csv('submission.csv', index=False)
    print(f"Submission saved! Total rows: {len(df)}")
    print(f"Expected: {len(test_ids)} images × {NUM_CLASSES} classes = {len(test_ids) * NUM_CLASSES}")
    
    return df

def load_best_model(checkpoint_path, device='cuda'):
    """Load best model from checkpoint"""
    model = init_model(device)
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    print(f"Loaded model from epoch {checkpoint['epoch']} with mIoU {checkpoint['best_miou']:.4f}")
    return model

print("Inference utilities ready.")

In [ ]:
# Cell 14: Visualisasi Prediksi (Sanity Check)
import matplotlib.pyplot as plt

def visualize_predictions(model, test_img_dir, test_ids, device, num_samples=5, 
                          save_path=None):
    """
    Visualisasi prediksi pada test set untuk sanity check
    """
    model.eval()
    
    infer_transform = A.Compose([
        A.Resize(512, 512),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])
    
    # Color mapping untuk visualisasi
    vis_colors = {
        0: [0, 0, 0],        # background - hitam
        1: [255, 0, 0],      # building flooded - merah
        2: [139, 69, 19],    # building non-flooded - coklat
        3: [0, 255, 0],      # grass - hijau
        4: [0, 255, 255],    # pool - cyan
        5: [255, 165, 0],    # road flooded - orange
        6: [128, 128, 128],  # road non-flooded - abu-abu
        7: [0, 100, 0],      # tree - hijau tua
        8: [255, 255, 0],    # vehicle - kuning
        9: [0, 0, 255]       # water - biru
    }
    
    def mask_to_rgb(mask):
        rgb = np.zeros((*mask.shape, 3), dtype=np.uint8)
        for cls, color in vis_colors.items():
            rgb[mask == cls] = color
        return rgb
    
    samples = np.random.choice(test_ids, min(num_samples, len(test_ids)), replace=False)
    
    fig, axes = plt.subplots(len(samples), 3, figsize=(15, 4 * len(samples)))
    if len(samples) == 1:
        axes = axes.reshape(1, -1)
    
    for i, img_id in enumerate(samples):
        # Load image
        img_path = os.path.join(test_img_dir, f"{img_id}.jpg")
        img = np.array(PILImage.open(img_path).convert('RGB'))
        
        # Predict
        transformed = infer_transform(image=img)['image'].unsqueeze(0).to(device)
        with torch.no_grad():
            logits = model(transformed)
            pred = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()
        
        # Resize pred back to original
        pred_img = PILImage.fromarray(pred.astype(np.uint8))
        try:
            pred_img = pred_img.resize((640, 480), PILImage.Resampling.NEAREST)
        except AttributeError:
            pred_img = pred_img.resize((640, 480), Image.NEAREST)
        pred_resized = np.array(pred_img)
        
        # Convert to RGB for visualization
        pred_rgb = mask_to_rgb(pred_resized)
        
        # Plot
        axes[i, 0].imshow(img)
        axes[i, 0].set_title(f"Original: {img_id}", fontsize=10)
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(pred_rgb)
        axes[i, 1].set_title("Prediction", fontsize=10)
        axes[i, 1].axis('off')
        
        axes[i, 2].imshow(img)
        axes[i, 2].imshow(pred_rgb, alpha=0.5)
        axes[i, 2].set_title("Overlay", fontsize=10)
        axes[i, 2].axis('off')
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved visualization to {save_path}")
    plt.show()
    
    return samples

print("Visualization utilities ready.")

In [ ]:
# Cell 15: Validasi Distribusi Prediksi
def analyze_submission_predictions(submission_path='submission.csv'):
    """
    Analisis distribusi prediksi untuk sanity check
    """
    df = pd.read_csv(submission_path)
    
    print("="*60)
    print("SUBMISSION PREDICTION ANALYSIS")
    print("="*60)
    
    # Per-class statistics
    print("\nPer-class detection:")
    print("-"*40)
    for class_id in range(NUM_CLASSES):
        if class_id in EMPTY_CLASSES:
            continue
        class_rows = df[df['id'].str.endswith(f'_{class_id}')]
        non_empty = class_rows[class_rows['encoded_pixels'] != '']
        pct = len(non_empty) / len(class_rows) * 100
        print(f"  {CLASS_NAMES[class_id]:<20}: {len(non_empty):>3}/{len(class_rows)} images ({pct:.1f}%)")
    
    # Check rare classes
    print("\n🔍 Rare classes check:")
    for class_id in RARE_CLASSES:
        class_rows = df[df['id'].str.endswith(f'_{class_id}')]
        non_empty = class_rows[class_rows['encoded_pixels'] != '']
        if len(non_empty) == 0:
            print(f"  WARNING: {CLASS_NAMES[class_id]} tidak terdeteksi di SATU PUN gambar!")
        elif len(non_empty) < 20:
            print(f"  CAUTION: {CLASS_NAMES[class_id]} hanya terdeteksi di {len(non_empty)} gambar")
        else:
            print(f"  {CLASS_NAMES[class_id]}: terdeteksi di {len(non_empty)} gambar")
    
    # Sample images with vehicle detection
    vehicle_ids = df[df['id'].str.endswith('_8') & (df['encoded_pixels'] != '')]['id'].str.split('_').str[0].tolist()
    if vehicle_ids:
        print(f"\n🚗 Contoh gambar dengan vehicle: {vehicle_ids[:10]}")
    
    # Sample images with water detection
    water_ids = df[df['id'].str.endswith('_9') & (df['encoded_pixels'] != '')]['id'].str.split('_').str[0].tolist()
    if water_ids:
        print(f"Contoh gambar dengan water: {water_ids[:10]}")
    
    # Total unique images with any non-background prediction
    all_ids = set()
    for class_id in range(NUM_CLASSES):
        if class_id in EMPTY_CLASSES:
            continue
        class_ids = df[df['id'].str.endswith(f'_{class_id}') & (df['encoded_pixels'] != '')]['id'].str.split('_').str[0].tolist()
        all_ids.update(class_ids)
    
    print(f"\nTotal gambar dengan setidaknya 1 kelas non-background: {len(all_ids)}/{len(df)//10}")
    
    return df

print("Analysis utilities ready.")

In [ ]:
# Cell 16: Validasi Format Submission (FIXED - dengan Pillow version check)
def validate_submission(submission_path='submission.csv'):
    """Validate submission format sesuai aturan kompetisi"""
    df = pd.read_csv(submission_path)
    
    print("="*60)
    print("SUBMISSION FORMAT VALIDATION")
    print("="*60)
    
    # Check columns
    assert list(df.columns) == ['id', 'encoded_pixels'], f"Columns: {df.columns}"
    print("Columns: 'id', 'encoded_pixels'")
    
    # Check 4470 rows (447 images × 10 classes)
    expected_rows = 447 * 10
    assert len(df) == expected_rows, f"Expected {expected_rows} rows, got {len(df)}"
    print(f"Total rows: {len(df)} (expected {expected_rows})")
    
    # Check ID format
    sample_id = df['id'].iloc[0]
    assert '_' in sample_id, f"ID format error: {sample_id}"
    print(f"ID format: '{sample_id}' (image_class)")
    
    # Check each image has exactly 10 classes
    img_ids = set([x.split('_')[0] for x in df['id']])
    expected_img_count = 447
    assert len(img_ids) == expected_img_count, f"Expected {expected_img_count} images, got {len(img_ids)}"
    
    for img_id in list(img_ids)[:5]:  # Check first 5 as sample
        class_ids = [int(x.split('_')[1]) for x in df['id'] if x.startswith(f"{img_id}_")]
        assert len(class_ids) == 10, f"Image {img_id} has {len(class_ids)} classes, expected 10"
        assert set(class_ids) == set(range(10)), f"Image {img_id} missing classes: {set(range(10)) - set(class_ids)}"
    print(f"Each of {len(img_ids)} images has exactly 10 classes (0-9)")
    
    # Check RLE format
    invalid_rows = []
    for idx, row in df.iterrows():
        if pd.notna(row['encoded_pixels']) and row['encoded_pixels'] != '':
            parts = row['encoded_pixels'].split()
            if len(parts) % 2 != 0:
                invalid_rows.append(idx)
    
    if invalid_rows:
        print(f"RLE format error at rows: {invalid_rows}")
    else:
        print("All RLE encodings have valid format (even number of integers)")
    
    # Check empty classes (2 and 6)
    for class_id in EMPTY_CLASSES:
        class_rows = df[df['id'].str.endswith(f'_{class_id}')]
        non_empty = class_rows[class_rows['encoded_pixels'] != '']
        if len(non_empty) > 0:
            print(f"WARNING: Class {class_id} ({CLASS_NAMES[class_id]}) has {len(non_empty)} non-empty predictions")
        else:
            print(f"Class {class_id} ({CLASS_NAMES[class_id]}): all empty (correct)")
    
    print("\n" + "="*60)
    print("SUBMISSION FORMAT IS VALID!")
    print("="*60)
    return True

print("Validation utilities ready.")